In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import string
#

In [51]:
df = pd.read_csv('train.txt',sep=';',header=None , names=['text','emotion'])

In [52]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [53]:
df.isnull().sum()

,0
text,0
emotion,0


In [54]:
uiq_emotion = df['emotion'].unique()
emotion_dict = {}
i = 0
for emo in uiq_emotion:
  emotion_dict[emo] = i
  i += 1


df['emotion'] = df['emotion'].map(emotion_dict)


In [55]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [56]:
df['text'] = df['text'].str.lower()

In [57]:
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

In [58]:
df['text'] = df['text'].apply(remove_punc)

In [59]:
df['text'] = df['text'].str.replace(r'/d+','',regex=True)

In [60]:
df['text'] = df['text'].str.replace(r'[^\x00-\x7F]+','',regex=True)

In [61]:
import nltk

In [62]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [63]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [64]:
stop_words = set(stopwords.words('english'))

In [65]:
def remove(txt):
  cleaned = []
  world = word_tokenize(txt)
  for i in world :
    if not i in stop_words:
      cleaned.append(i)

  return " ".join(cleaned)

In [66]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [67]:
df['text'] = df['text'].apply(remove)

In [68]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( df['text'], df['emotion'], test_size=0.20, random_state=42)

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [70]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [71]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


In [72]:
naive_bayes = MultinomialNB()
naive_bayes.fit(X_train_bow, y_train)


MultinomialNB()

In [73]:
y_pred_nb = naive_bayes.predict(X_test_bow)
accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)

Accuracy: 0.7678125


In [74]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [75]:
naive_bayes_2 = MultinomialNB()
naive_bayes_2.fit(X_train_tfidf, y_train)

MultinomialNB()

In [76]:
y_pred_nb_2 = naive_bayes_2.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_nb_2)
print("Accuracy:", accuracy)

Accuracy: 0.6609375


In [77]:
from sklearn.linear_model import LogisticRegression
model_log = LogisticRegression(max_iter=1000)

In [78]:
model_log.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [79]:
y_pred_log = model_log.predict(X_test_tfidf)

In [80]:
accuracy = accuracy_score(y_test, y_pred_log)
print("Accuracy:", accuracy)

Accuracy: 0.8615625


In [81]:
print(df['emotion'].unique())
print(emotion_dict)

[0 1 2 3 4 5]
{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [82]:
import joblib

joblib.dump(model_log, 'model_lr.pkl')
joblib.dump(emotion_dict, 'emotion_labels.pkl')

# Download
from google.colab import files
files.download('model_lr.pkl')
files.download('emotion_labels.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [83]:
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

from google.colab import files
files.download('tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>